In [2]:
from dotenv import load_dotenv
# Load .env once at startup
load_dotenv()

True

# **Environment Set-up**
### 1. Ensure the Python Interpreter is pointing to the virtual environment with the correct packages:
The python virtual environment is available as a bash script here: .  Or you can create your own virtual environment

In [3]:
import sys, os
# print the absolute path to the executable binary for the Python interpreter
# MacOS/Linux: command + shift + p -> "Python: Select Interpreter" -> select the correct environment
# Windows: ctrl + shift + p -> "Python: Select Interpreter" -> select the correct environment
print(sys.executable)
# print the file system path to the root directory of hte currently active Python Virtual Environment
# VIRTUAL_ENV is set by the virtual environment's `activate` script, terminal command: 'source <path/to>/.venv/bin/activate'
print(os.environ.get("VIRTUAL_ENV"))

/Users/eibeck/Projects/gary-livelab/lib/.venv/bin/python
/Users/eibeck/Projects/gary-livelab/lib/.venv


In [ ]:
'''
import requests, certifi, ssl, os
url="https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json"
print("python:", ssl.OPENSSL_VERSION)
print("certifi:", certifi.where())
print("SSL_CERT_FILE:", os.environ.get("SSL_CERT_FILE"))
print("REQUESTS_CA_BUNDLE:", os.environ.get("REQUESTS_CA_BUNDLE"))
r = requests.head(url, timeout=30, allow_redirects=True)
print("status:", r.status_code)
print("final_url:", r.url)
'''

python: OpenSSL 3.0.16 11 Feb 2025
certifi: /Users/eibeck/Projects/gary-livelab/lib/.venv/lib/python3.13/site-packages/certifi/cacert.pem
SSL_CERT_FILE: /Users/eibeck/oracle_wallet/ewallet.pem
REQUESTS_CA_BUNDLE: None
status: 404
final_url: https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json


# **Notebook Objectives**
## Complete four of five Steps in Oracle Retrieval-Augmented Generation (RAG)
### 1. Document Ingestion & Chunking:
Unstructured data (PDFs, text files) is uploaded to OCI Object Storage and processed into smaller chunks.
### 2. Embedding Generation (Embeddings Model):
The chunks are sent to an embedding model to convert text into vector embeddings.
### 3. Vector Storage:
The vectors and original text chunks are stored in an Oracle Autonomous Database table using the native VECTOR data type.
### 4. Semantic Search (Retrieval):
User questions are vectorized, and Oracle Vector Search finds similar documents.
### 5. Generation (LLM):
(This step was not included in the Jupyter Notebook)The context (top results pulled from Step 4), along with the user question, is sent to a Generative AI Model (LLM) to produce an accurate answer.

(https://www.oracle.com/artificial-intelligence/generative-ai/retrieval-augmented-generation-rag/)

## Set-up Oracle AI Database Connection
### Prerequisites
* Administrative privileges to an Autonomous AI Database (recommend use free version)
* Download the database wallet
* Ensure the database Network settings allow calls from your CIDR (0.0.0.0/0) range or your IP address
### Code Block #1: Connect using python-oracledb
1. Unzip the wallet.
2. Set the Environment Variables used for connecting to the database: (`TNS_ADMIN`), and pool variables to connect to the database: (`DB_USER`, `DB_PASSWORD`, `CONNECT_STRING`)
3. Initialize a pool variable.  (To make the database connection available across different code blocks use a global connection pool. The `pool` variable will be used to create connections on-demand for each query.)

### Code Block #1: Run the first cell to test the connection.
(Subsequent database calls will open and close a database connection with each operation.)


In [30]:
from pathlib import Path
import os

wallet_dir = Path("~/oracle_wallet").expanduser()
print("wallet_dir:", wallet_dir)
#print("TNS_ADMIN:", os.environ.get("TNS_ADMIN"))
print("tnsnames exists:", (wallet_dir / "tnsnames.ora").exists())
print("cwallet exists:", (wallet_dir / "cwallet.sso").exists())

wallet_dir: /Users/eibeck/oracle_wallet
tnsnames exists: True
cwallet exists: True


In [4]:
'''
Your notebook is using oracledb.connect() correctly.
But Python-oracledb Thin mode uses Python/OpenSSL trust handling.
SQL Developer uses Java truststores and often succeeds in environments where Python Thin fails.
“Self-signed certificate in certificate chain” strongly suggests a corporate/security proxy CA (or missing CA trust) between your machine and ADB endpoint.
Best fix path (most reliable)
Use Thick mode (Instant Client), which usually behaves more like SQL Developer for Oracle TLS/wallet handling.

In your notebook (before first connect):
'''
import oracledb

# oracledb.init_oracle_client(lib_dir="/absolute/path/to/instantclient")
# print("thin mode?", oracledb.is_thin_mode())  # should print False

In [ ]:
# Code Block #1
# Follow driver installation and setup instructions here:
# https://www.oracle.com/database/technologies/appdev/python/quickstartpython.html

import zipfile
import os
from pathlib import Path

# Unzip wallet if not already
wallet_zip = Path('~/Downloads/Wallet_livelab.zip').expanduser()
wallet_dir = Path('~/oracle_wallet').expanduser()
if not wallet_dir.exists():
    with zipfile.ZipFile(wallet_zip, 'r') as zip_ref:
        zip_ref.extractall(wallet_dir)
# set TNS_ADMIN to the wallet directory
# this tells the Oracle client libraries where to find the network configuration files (like tnsnames.ora)
# without setting TNS_ADMIN, the library wouldn't know where to find these configuration files and would fail to connect to the database using the wallet
os.environ['TNS_ADMIN'] = str(wallet_dir)

# Replace USER_NAME, PASSWORD with your username and password
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
# Use wallet
CONNECT_STRING = os.getenv("CONNECT_STRING")




# Create global connection function to be used throughout the notebook
def get_connection():
    return oracledb.connect(
        config_dir=str(wallet_dir),
        user=DB_USER,
        password=DB_PASSWORD,
        dsn=CONNECT_STRING
    )

def test_connection():
	try:
		with get_connection() as connection:
			with connection.cursor() as cursor:
				cursor.execute("SELECT 1 FROM DUAL")
				result = cursor.fetchone()
				if result:
					print(f"Connected successfully! Query result: {result[0]}")
	except oracledb.Error as e:
		print('Oracle error:', e)
		print('Args:', getattr(e, 'args', None))
	except Exception as e:
		import traceback
		traceback.print_exc()





# Import most of the necessary libraries into your notebook

### Code Block #2: Import the necessary libraries

In [ ]:
# Code Block #1 (Thin mode + SSL troubleshooting)
# Replace your current Code Block #1 with this entire block.

import os
import zipfile
from pathlib import Path
import ssl
import certifi
import oracledb

# --- Wallet setup ---
wallet_zip = Path("~/Downloads/Wallet_livelab.zip").expanduser()
wallet_dir = Path("~/oracle_wallet").expanduser()

if not wallet_dir.exists():
    if not wallet_zip.exists():
        raise FileNotFoundError(f"Wallet ZIP not found: {wallet_zip}")
    with zipfile.ZipFile(wallet_zip, "r") as zip_ref:
        zip_ref.extractall(wallet_dir)

# Required env vars for wallet-based connect
os.environ["TNS_ADMIN"] = str(wallet_dir)

# Thin-mode cert trust: point Python/OpenSSL at wallet PEM chain
# If your org has TLS interception, this helps include wallet cert chain.
wallet_pem = wallet_dir / "ewallet.pem"
if wallet_pem.exists():
    os.environ["SSL_CERT_FILE"] = str(wallet_pem)

# --- Connection settings ---
#DB_USER = 
#DB_PASSWORD = 
#CONNECT_STRING = 

def get_connection():
    return oracledb.connect(
        config_dir=str(wallet_dir),
        user=DB_USER,
        password=DB_PASSWORD,
        dsn=CONNECT_STRING
    )

def test_connection():
    print("=== Runtime Diagnostics ===")
    print("oracledb thin mode:", oracledb.is_thin_mode())  # expected True
    print("TNS_ADMIN:", os.environ.get("TNS_ADMIN"))
    print("SSL_CERT_FILE:", os.environ.get("SSL_CERT_FILE"))
    print("wallet_dir exists:", wallet_dir.exists(), wallet_dir)
    print("tnsnames.ora exists:", (wallet_dir / "tnsnames.ora").exists())
    print("sqlnet.ora exists:", (wallet_dir / "sqlnet.ora").exists())
    print("ewallet.pem exists:", wallet_pem.exists())
    print("cwallet.sso exists:", (wallet_dir / "cwallet.sso").exists())
    print("certifi.where():", certifi.where())
    print("OpenSSL default verify paths:", ssl.get_default_verify_paths())
    print("===========================")

    try:
        with get_connection() as connection:
            with connection.cursor() as cursor:
                cursor.execute("SELECT 1 FROM DUAL")
                result = cursor.fetchone()
                print(f"Connected successfully! Query result: {result[0]}")
                return True
    except oracledb.Error as e:
        print("Oracle error:", e)
        print("Args:", getattr(e, "args", None))
        return False
    except Exception as e:
        import traceback
        traceback.print_exc()
        return False


In [34]:
# Run test
test_connection()

=== Runtime Diagnostics ===
oracledb thin mode: True
TNS_ADMIN: /Users/eibeck/oracle_wallet
SSL_CERT_FILE: /Users/eibeck/oracle_wallet/ewallet.pem
wallet_dir exists: True /Users/eibeck/oracle_wallet
tnsnames.ora exists: True
sqlnet.ora exists: True
ewallet.pem exists: True
cwallet.sso exists: True
certifi.where(): /Users/eibeck/Projects/gary-livelab/lib/.venv/lib/python3.13/site-packages/certifi/cacert.pem
OpenSSL default verify paths: DefaultVerifyPaths(cafile='/Users/eibeck/oracle_wallet/ewallet.pem', capath=None, openssl_cafile_env='SSL_CERT_FILE', openssl_cafile='/Library/Frameworks/Python.framework/Versions/3.13/etc/openssl/cert.pem', openssl_capath_env='SSL_CERT_DIR', openssl_capath='/Library/Frameworks/Python.framework/Versions/3.13/etc/openssl/certs')
Connected successfully! Query result: 1


True

In [35]:
# Code Block #2
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from PyPDF2 import PdfReader

# Choose a local .pdf document to tokenize & vectorize
### Code Block #3: Choose a .pdf file that is predominately text.

In [36]:
# Code Block #3
PDF_PATH = Path('~/Documents/oil_paints.pdf').expanduser()
assert PDF_PATH.exists(), f'PDF not found at {PDF_PATH}'

In [46]:
# since SSL_CERT_FILE is set to the wallet PEM, which doesn't include the full cert chain, we need to remove it to allow certifi/OpenSSL to use the default CA bundle for Hugging Face requests
os.environ.pop("SSL_CERT_FILE", None)   # removes it if present

'/Users/eibeck/oracle_wallet/ewallet.pem'

# Use SentenceTransformer(string) Constructor to instantiate embedding model
## (Post Set-up Instructions begin)
Creating the embedding_model object takes ~100MB of storage on heap; Recommended RAM >1GB.  Choose a different model, if desired, but more complex models will have additional parameters & take more time to load & execute

### Code Block #4: instantiate embedding_model object


In [ ]:
# Code Block #4
# 'all-MiniLM-L6-v2' is a smaller, faster model that still provides good performance for many tasks, including semantic search. It is
# a good choice for local embedding generation when you want a balance between speed and accuracy. (https://huggingface.co/models?library=sentence-transformers)
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
# The SentenceTransformer() constructor instantiates the embedding_model object, which will be used to encode text into embeddings.
# Setting the variable local_files_only to True forces the library to look for model files on the local filesystem and not attempt
# to download them from the Hugging Face Hub if found, which is necessary in environments without internet access and pulls the object form your
# local cache directory (usually ~/.cache/huggingface/sentence_transformers/) where the model files were downloaded during the initial setup with internet access.
try:
    model = SentenceTransformer("all-MiniLM-L6-v2", local_files_only=True)
except Exception:
    # first-time setup: download (with network)
    model = SentenceTransformer("all-MiniLM-L6-v2")

# 'Warning: You are sending unauthenticated requests to Hugging Face Hub' can be ignored. Every time you instantiate SentenceTransformer("model-name"),
# the library checks the HF Hub to see if a newer version of the model files exist.

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6003.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [49]:
# reset teh SSL_CERT_FILE env var to the wallet PEM for any subsequent code that needs to connect to the database with the wallet
os.environ["SSL_CERT_FILE"] = str(wallet_pem)

# Read the .pdf file
### Code Block #5: Instantiate the pdf_pages object with text

In [11]:
# Code Block #5
def load_pdf_pages(pdf_path: Path) -> List[Dict]:
    """Return a list of {'page_number': int, 'text': str} for each PDF page."""
    reader = PdfReader(str(pdf_path))
    pages = []
    for idx, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        pages.append({'page_number': idx, 'text': text.strip()})
    return pages

pdf_pages = load_pdf_pages(PDF_PATH)
total_chars = sum(len(page['text']) for page in pdf_pages)
print(f'Loaded {len(pdf_pages)} pages (~{total_chars:,} characters).')

Loaded 92 pages (~167,826 characters).


# Generate Embeddings
### Code Block #6: Chunk the text, add overlap to assist with context, save pages to provide future reference, create Embeddings

In [15]:
# Code Block #6
def chunk_text(text: str, max_chars: int = 900, overlap: int = 150) -> List[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        chunks.append(text[start:end].strip())
        if end == len(text):
            break
        start = max(0, end - overlap)
    return [chunk for chunk in chunks if chunk]

def prepare_chunks(pages: List[Dict]) -> List[Dict]:
    chunk_records = []
    for page in pages:
        for chunk_idx, chunk in enumerate(chunk_text(page['text'])):
            chunk_records.append({
                'page_number': page['page_number'],
                'chunk_index': chunk_idx,
                'text': chunk
            })
    return chunk_records

def embed_chunks(chunks: List[Dict]) -> Tuple[np.ndarray, List[Dict]]:
    texts = [chunk['text'] for chunk in chunks]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    return embeddings, chunks

chunks = prepare_chunks(pdf_pages)
chunk_embeddings, chunk_metadata = embed_chunks(chunks)
print(f'Generated {len(chunk_embeddings)} embeddings with dimension {chunk_embeddings.shape[1]}.')

Batches: 100%|██████████| 8/8 [00:00<00:00,  9.16it/s]

Generated 250 embeddings with dimension 384.


# Create Table in the Database
### Code Block #7: Connect to the Oracle AI Database and create the table to store vectors along with the page number where the information was gathered

In [16]:
# Code Block #7
# Create the oil_paint_chunks table if it doesn't exist
with get_connection() as connection:
    with connection.cursor() as cursor:
        # Check if table exists
        cursor.execute("SELECT table_name FROM user_tables WHERE table_name = 'OIL_PAINT_CHUNKS'")
        if not cursor.fetchone():
            cursor.execute("""
                CREATE TABLE oil_paint_chunks (
                    chunk_id     NUMBER GENERATED ALWAYS AS IDENTITY,
                    page_number  NUMBER,
                    chunk_index  NUMBER,
                    text_chunk   CLOB,
                    embedding    VECTOR(384)
                )
            """)
            print("Table 'oil_paint_chunks' created successfully.")
        else:
            cursor.execute("""
                TRUNCATE TABLE oil_paint_chunks;
            """)
            print("Table 'oil_paint_chunks' already exists.")
        connection.commit()

Table 'oil_paint_chunks' already exists.


# Insert embeddings into the database
### Code Block #8: Store the vector data into the database

In [17]:
# Code Block #8
insert_sql = "INSERT INTO oil_paint_chunks (page_number, chunk_index, text_chunk, embedding) VALUES (:page_number, :chunk_index, :text_chunk, VECTOR(:embedding))"

with get_connection() as connection:
    with connection.cursor() as cursor:
        cursor.setinputsizes(embedding=oracledb.DB_TYPE_CLOB)
        data = [
            {
                'page_number': meta['page_number'],
                'chunk_index': meta['chunk_index'],
                'text_chunk': meta['text'],
                'embedding': '[' + ','.join(map(str, embedding.tolist())) + ']'
            }
            for meta, embedding in zip(chunk_metadata, chunk_embeddings)
        ]
        for params in data:
            cursor.execute(insert_sql, params)
        connection.commit()
    print(f"Inserted {len(chunk_metadata)} chunks into the database.")

Inserted 250 chunks into the database.


# Semantic Search function with embedding_model
### Code Block #9: Use cos_sim to find closest page and text given a query string

In [18]:
# Code Block #9
# This notebook does not contain any natural language to SQL conversion.  It performs semantic search entirely through the local 
# embedding model and cosine similarity.  The best_page_for() function returns the page number of the most relevant page for a given query.
def semantic_search(query: str, top_k: int = 3) -> List[Dict]:
    query_emb = embedding_model.encode(query, convert_to_numpy=True)
    scores = cos_sim(query_emb, chunk_embeddings)[0]
    top_k = min(top_k, len(scores))
    #top_indices = np.argsort(scores)[::-1][:top_k]
    top_indices = np.argsort(-scores)[:top_k]
    top_scores = scores[top_indices]
    matches = []
    for score, idx in zip(top_scores, top_indices):
        meta = chunk_metadata[idx]
        matches.append({
            'score': float(score),
            'page_number': meta['page_number'],
            'chunk_index': meta['chunk_index'],
            'text': meta['text']
        })
    return matches

def best_page_for(query: str) -> int:
    result = semantic_search(query, top_k=1)
    return result[0]['page_number'] if result else -1

# Create a query string
### Code Block #10: Modify the string if desired & test `semantic_seach()`

In [19]:
# Code Block #10
query = "What pigments are recommended for glazing?"
matches = semantic_search(query, top_k=2)
print('Top match likely on page:', best_page_for(query))
for match in matches:
    print(f"Page {match['page_number']} (score={match['score']:.3f}): {match['text'][:200]}...")

Top match likely on page: 76
Page 76 (score=0.497): olour into existing, still wet 
layers.  The technique can be used to bring great immediacy and interest tothe image.  It also can be used as a technique for blending, and can beaccomplished with the ...
Page 12 (score=0.492): sto and glazing can be done in considerably lesstime than when working in traditional oils.  The colours are ideal forworking outdoors.  Consistent drying times across the range removes theusual restr...


# Semantic Search function with embedding_model and Oracle AI Vector Search
### Code Block #11: VECTOR_SEARCH can compare with COSINE, EUCLIDEAN, DOT, MANHATTAN...
(https://docs.oracle.com/en/database/oracle/oracle-database/26/sqlrf/vector_distance.html#GUID-BA4BCFB2-D905-43DC-87B0-E53522CF07B7)

In [24]:
# Code Block #11
# Oracle-based semantic search using VECTOR_COSINE_DISTANCE
def oracle_semantic_search(query: str, top_k: int = 3) -> List[Dict]:
    query_emb = embedding_model.encode(query, convert_to_numpy=True)
    query_vector_str = '[' + ','.join(map(str, query_emb.tolist())) + ']'

    matches = []
    with get_connection() as connection:
        with connection.cursor() as cursor:
            sql = f"""
            SELECT page_number,
                   chunk_index,
                   text_chunk,
                   VECTOR_DISTANCE(embedding, VECTOR(:query_vector), COSINE) AS distance
            FROM oil_paint_chunks
            ORDER BY VECTOR_DISTANCE(embedding, VECTOR(:query_vector), COSINE)
            FETCH FIRST {top_k} ROWS ONLY
            """
            cursor.execute(sql, {'query_vector': query_vector_str})

            for page_number, chunk_index, text_chunk, distance in cursor:
                text_content = text_chunk.read() if hasattr(text_chunk, 'read') else str(text_chunk)
                matches.append({
                    'score': 1.0 - distance,
                    'page_number': page_number,
                    'chunk_index': chunk_index,
                    'text': text_content
                })

    return matches


def oracle_best_page_for(query: str) -> int:
    result = oracle_semantic_search(query, top_k=1)
    return result[0]['page_number'] if result else -1


In [25]:
test_connection()

=== Runtime Diagnostics ===
oracledb thin mode: True
TNS_ADMIN: /Users/eibeck/oracle_wallets
SSL_CERT_FILE: /Users/eibeck/oracle_wallets/ewallet.pem
wallet_dir exists: True /Users/eibeck/oracle_wallets
tnsnames.ora exists: True
sqlnet.ora exists: True
ewallet.pem exists: True
cwallet.sso exists: True
certifi.where(): /Users/eibeck/Projects/gary-livelab/lib/.venv/lib/python3.13/site-packages/certifi/cacert.pem
OpenSSL default verify paths: DefaultVerifyPaths(cafile='/Users/eibeck/oracle_wallets/ewallet.pem', capath=None, openssl_cafile_env='SSL_CERT_FILE', openssl_cafile='/Library/Frameworks/Python.framework/Versions/3.13/etc/openssl/cert.pem', openssl_capath_env='SSL_CERT_DIR', openssl_capath='/Library/Frameworks/Python.framework/Versions/3.13/etc/openssl/certs')
Connected successfully! Query result: 1


True

# Create a query string
### Code Block #12: Modify the string if desired & test `oracle_semantic_seach()`

In [50]:
# Code Block #12
query = "What pigments are recommended for glazing?"
matches = oracle_semantic_search(query, top_k=2)
print('Oracle-based top match likely on page:', oracle_best_page_for(query))
for match in matches:
    print(f"Page {match['page_number']} (score={match['score']:.3f}): {match['text'][:200]}...")


Oracle-based top match likely on page: 76
Page 76 (score=0.497): olour into existing, still wet 
layers.  The technique can be used to bring great immediacy and interest tothe image.  It also can be used as a technique for blending, and can beaccomplished with the ...
Page 12 (score=0.492): sto and glazing can be done in considerably lesstime than when working in traditional oils.  The colours are ideal forworking outdoors.  Consistent drying times across the range removes theusual restr...


# Notebook Summary
## Key Steps in Oracle Retrieval-Augmented Generation (RAG)
### 1. Document Ingestion & Chunking:
Unstructured data (PDFs, text files) is uploaded to OCI Object Storage and processed into smaller chunks.
### 2. Embedding Generation (Embeddings Model):
The chunks are sent to an embedding model to convert text into vector embeddings.
### 3. Vector Storage:
The vectors and original text chunks are stored in an Oracle Autonomous Database table using the native VECTOR data type.
### 4. Semantic Search (Retrieval):
User questions are vectorized, and Oracle Vector Search finds similar documents.
### 5. Generation (LLM):
(This step was not included in the Jupyter Notebook)The context (top results pulled from Step 4), along with the user question, is sent to a Generative AI Model (LLM) to produce an accurate answer.

(https://www.oracle.com/artificial-intelligence/generative-ai/retrieval-augmented-generation-rag/)



# Create a Natural Language to SQL object
### Code Block #13: This takes several GB of RAM and may take a few minutes to complete.  There is no follow-on use for this code, but may be used to create SQL queries to interrogate your Oracle AI Database, for `fun`!

In [ ]:
# Code Block #13
'''
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

class LocalSqlGenerator:
    def __init__(self, model_name="chatdb/natural-sql-7b"):
        # Constructor loads model and tokenizer into the heap
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto", # Automatically uses GPU if available
            torch_dtype="auto"
        )
        self.generator = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer
        )

    def generate_query(self, question, schema):
        prompt = f"Schema: {schema}\nQuestion: {question}\nSQL:"
        result = self.generator(prompt, max_new_tokens=200)
        return result[0]['generated_text']

# Usage: Instantiate once, keep in heap
sql_tool = LocalSqlGenerator()
query = sql_tool.generate_query("Show me total sales by month", "Table: Sales (Date, Amount)")

print(query)
'''


'\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, pipeline\n\nclass LocalSqlGenerator:\n    def __init__(self, model_name="chatdb/natural-sql-7b"):\n        # Constructor loads model and tokenizer into the heap\n        self.tokenizer = AutoTokenizer.from_pretrained(model_name)\n        self.model = AutoModelForCausalLM.from_pretrained(\n            model_name,\n            device_map="auto", # Automatically uses GPU if available\n            torch_dtype="auto"\n        )\n        self.generator = pipeline(\n            "text-generation",\n            model=self.model,\n            tokenizer=self.tokenizer\n        )\n\n    def generate_query(self, question, schema):\n        prompt = f"Schema: {schema}\nQuestion: {question}\nSQL:"\n        result = self.generator(prompt, max_new_tokens=200)\n        return result[0][\'generated_text\']\n\n# Usage: Instantiate once, keep in heap\nsql_tool = LocalSqlGenerator()\nquery = sql_tool.generate_query("Show me total sales by mont